# PMI (Pointwise Mutual Information)

El **PMI (Pointwise Mutual Information)**, o Información Mutua Puntual, es otra métrica fundamental en el Procesamiento de Lenguaje Natural (y la teoría de la información en general). 

Mientras que TF-IDF se enfoca en la importancia de una palabra respecto a un documento, el **PMI se enfoca en la relación o asociación entre dos palabras**. Su objetivo es responder: **¿Qué tan probable es que estas dos palabras aparezcan juntas en comparación con lo que esperaríamos si aparecieran por pura casualidad?**

Aquí tienes la explicación conceptual, el ejemplo manual y la implementación en Python.

## 1. La Teoría: ¿Por qué PMI?

Imagina las palabras "San" y "Francisco". Si lees "San" en un texto, hay una probabilidad muy alta de que la siguiente palabra sea "Francisco" (o "Juan", "Diego", etc.). Estas palabras forman una **colocación**. El PMI mide matemáticamente esa fuerza de atracción.

Para calcularlo, comparamos dos escenarios:
1.  **La probabilidad conjunta $P(x, y)$:** La probabilidad de que las palabras $x$ e $y$ aparezcan juntas (por ejemplo, "inteligencia artificial").
2.  **La probabilidad independiente $P(x) \cdot P(y)$:** La probabilidad de que aparezcan juntas si su aparición fuera un evento totalmente aleatorio e independiente.

### La Fórmula Matemática

El PMI de un par de palabras (o eventos) $x$ e $y$ se define usando un logaritmo (generalmente en base 2):

$$PMI(x, y) = \log_2 \left( \frac{P(x, y)}{P(x) \cdot P(y)} \right)$$

**¿Cómo interpretar el resultado?**
*   **PMI > 0:** Las palabras aparecen juntas *más* de lo esperado por azar. Tienen una asociación positiva (ej. "Puerto" y "Rico").
*   **PMI = 0:** Las palabras son totalmente independientes. Su co-ocurrencia es pura coincidencia estadística (ej. "el" y "zapato").
*   **PMI < 0:** Las palabras aparecen juntas *menos* de lo esperado. Se evitan mutuamente (asociación negativa).

> **Nota técnica:** En NLP, calcular PMI negativo es problemático porque si dos palabras nunca aparecen juntas en el corpus, $P(x, y) = 0$, y el $\log_2(0)$ tiende a menos infinito. Por eso en la industria se usa el **PPMI (Positive PMI)**, que simplemente cambia todos los valores negativos a 0: $PPMI(x, y) = \max(PMI(x, y), 0)$.

## 2. Toy Example (Cálculo Manual)

Supongamos que tenemos un corpus de texto que, al dividirlo en pares de palabras contiguas (bigramas), tiene un total de **100 pares**.

Vamos a analizar el par $x =$ "inteligencia" e $y =$ "artificial".
Y vamos a compararlo con el par $a =$ "el" y $b =$ "gato".

**Datos de nuestro corpus de 100 pares:**
*   "inteligencia" aparece 5 veces $\rightarrow P(\text{inteligencia}) = 5/100 = 0.05$
*   "artificial" aparece 4 veces $\rightarrow P(\text{artificial}) = 4/100 = 0.04$
*   "inteligencia artificial" aparecen juntas 3 veces $\rightarrow P(\text{inteligencia, artificial}) = 3/100 = 0.03$

*   "el" aparece 20 veces $\rightarrow P(\text{el}) = 20/100 = 0.20$
*   "gato" aparece 5 veces $\rightarrow P(\text{gato}) = 5/100 = 0.05$
*   "el gato" aparecen juntas 1 vez $\rightarrow P(\text{el, gato}) = 1/100 = 0.01$

**Cálculo para "inteligencia artificial":**
$$PMI = \log_2 \left( \frac{0.03}{0.05 \times 0.04} \right) = \log_2 \left( \frac{0.03}{0.002} \right) = \log_2(15) \approx 3.91$$
*Resultado:* Un PMI alto y positivo. Tienen una fuerte dependencia semántica.

**Cálculo para "el gato":**
$$PMI = \log_2 \left( \frac{0.01}{0.20 \times 0.05} \right) = \log_2 \left( \frac{0.01}{0.01} \right) = \log_2(1) = 0$$
*Resultado:* Un PMI de 0. Son totalmente independientes estadísticamente; aparecen juntas en la proporción exacta que dictaría el puro azar considerando lo comunes que son en el texto.

## 3. Implementación en Python

Para calcular el PMI en Python de forma robusta, la herramienta estándar en el ecosistema tradicional de NLP es la librería `nltk` (Natural Language Toolkit).

Primero, si no tienes la librería, instálala:
```bash
pip install nltk
```

In [1]:
import nltk
from nltk.collocations import BigramCollocationFinder
from nltk.metrics import BigramAssocMeasures

# 1. Nuestro texto de ejemplo tokenizado (dividido en palabras)
corpus = [
    "la", "inteligencia", "artificial", "es", "el", "futuro", 
    "mucha", "gente", "estudia", "inteligencia", "artificial", 
    "el", "gato", "come", "mientras", "el", "perro", "duerme"
]

# 2. Inicializamos el buscador de bigramas (pares de palabras)
finder = BigramCollocationFinder.from_words(corpus)

# 3. Importamos las métricas de asociación
metricas = BigramAssocMeasures()

# (Opcional) Podemos filtrar para calcular el PMI solo de los pares 
# que aparezcan al menos N veces (útil en corpus grandes)
finder.apply_freq_filter(1)

# 4. Calculamos el PMI para todos los bigramas encontrados
# Esto devuelve una lista de tuplas: ( (palabra1, palabra2), score_pmi )
pmi_scores = finder.score_ngrams(metricas.pmi)

# 5. Mostramos los resultados ordenados de mayor a menor PMI
print("Top Bigramas por PMI:")
for bigrama, score in pmi_scores:
    print(f"{bigrama[0]} {bigrama[1]}: {score:.2f}")

Top Bigramas por PMI:
come mientras: 4.17
futuro mucha: 4.17
gato come: 4.17
gente estudia: 4.17
mucha gente: 4.17
perro duerme: 4.17
artificial es: 3.17
estudia inteligencia: 3.17
inteligencia artificial: 3.17
la inteligencia: 3.17
el futuro: 2.58
el gato: 2.58
el perro: 2.58
es el: 2.58
mientras el: 2.58
artificial el: 1.58
